# LightGBM: A Hands-On Lesson

LightGBM (Light Gradient Boosting Machine, from Microsoft) is a gradient-boosting library built for speed on large tables. Like XGBoost, it builds trees one after another, each correcting the errors of the last. Its key differences:

- **Leaf-wise tree growth** lets it reach low error with fewer trees, but it needs care to avoid overfitting.
- **Native categorical support.** Pandas `category` columns are handled directly, with no one-hot encoding.
- **Speed and memory efficiency** on large datasets.
- **The same objectives** as XGBoost for counts (`poisson`), positive skewed amounts (`gamma`), and Tweedie data (`tweedie`).

This lesson covers:

1. Setup and simulated data
2. Train/test split
3. A first regression model
4. Early stopping
5. Native categorical features
6. Poisson objective for claim frequency (with exposure as weights)
7. Gamma objective for claim severity
8. Monotonic constraints
9. Feature importance
10. The native API (`Dataset` and `lgb.train`)
11. Saving and loading models
12. Practice exercises, with solutions

The data is simulated, so nothing needs downloading. Run the cells in order.

## 1. Setup and simulated data

In [ ]:
import os

# Avoids a hang with OpenMP threading on some systems (including WSL2)
os.environ["OMP_NUM_THREADS"] = "1"

import numpy as np
import pandas as pd
import lightgbm as lgb

print("lightgbm", lgb.__version__)

We simulate policies with age, BMI, smoking status, a region (a categorical column), and exposure. Claims depend on these features.

In [ ]:
rng = np.random.default_rng(seed=5)
n = 20000

policies = pd.DataFrame({
    "age": rng.integers(18, 80, size=n),
    "bmi": rng.normal(27, 4.5, size=n).clip(15, 50).round(1),
    "smoker": rng.choice([0, 1], size=n, p=[0.8, 0.2]),
    "region": pd.Categorical(
        rng.choice(["urban", "suburban", "rural"], size=n, p=[0.4, 0.35, 0.25])
    ),
    "exposure": rng.uniform(0.25, 1.0, size=n),
})

region_effect = policies["region"].map({"urban": 0.25, "suburban": 0.05, "rural": 0.0}).astype(float)

true_rate = np.exp(
    -2.4
    + 0.012 * (policies["age"] - 40)
    + 0.4 * policies["smoker"]
    + region_effect
    + 0.01 * (policies["bmi"] - 27)
)
policies["claims"] = rng.poisson(true_rate * policies["exposure"])

policies.head()

In [ ]:
print("policies:", len(policies))
print("total claims:", policies["claims"].sum())
print("region dtype:", policies["region"].dtype)

Notice that `region` has the pandas `category` dtype. LightGBM uses this directly, without any encoding step.

## 2. Train/test split

We split by row, because each row is one policy. The test set stays untouched until the final check.

In [ ]:
from sklearn.model_selection import train_test_split

features = ["age", "bmi", "smoker", "region"]

train, test = train_test_split(policies, test_size=0.25, random_state=5)

X_train, y_train = train[features], train["claims"]
X_test, y_test = test[features], test["claims"]

print("train rows:", len(train), " test rows:", len(test))

## 3. A first regression model

LightGBM's scikit-learn wrapper is `LGBMRegressor`. It uses the same `fit`/`predict` pattern as the other models in these lessons.

In [ ]:
from sklearn.metrics import mean_absolute_error

model = lgb.LGBMRegressor(
    n_estimators=200,
    learning_rate=0.05,
    num_leaves=31,
    n_jobs=1,
    verbose=-1,
    random_state=5,
)
model.fit(X_train, y_train)

pred = model.predict(X_test)
print("test MAE (claim counts):", round(mean_absolute_error(y_test, pred), 4))

Key settings:

- **`n_estimators`**: the number of trees.
- **`learning_rate`**: how much each tree contributes.
- **`num_leaves`**: the main complexity control. LightGBM grows trees leaf-wise, so this matters more than `max_depth`. A common rule is to keep `num_leaves` below `2 ** max_depth`.
- **`n_jobs`**: threads to use. We set `1` for WSL2.
- **`verbose=-1`**: silences the training log.

## 4. Early stopping

Like XGBoost, LightGBM can stop when a validation score stops improving. In LightGBM this is a **callback**, passed in with `callbacks=`.

In [ ]:
X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=5)

model = lgb.LGBMRegressor(
    n_estimators=2000,
    learning_rate=0.05,
    num_leaves=15,
    n_jobs=1,
    verbose=-1,
    random_state=5,
)
model.fit(
    X_fit, y_fit,
    eval_X=X_val, eval_y=y_val,
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
)

print("best iteration:", model.best_iteration_)
print("test MAE:", round(mean_absolute_error(y_test, model.predict(X_test)), 4))

After early stopping, `best_iteration_` holds the number of trees used. Predictions use only those trees by default.

## 5. Native categorical features

LightGBM handles the `region` column directly, by finding splits on its categories. You don't need one-hot encoding, which can produce very wide tables. The model above already used it.

For comparison, here's the same model with `region` converted to plain integer codes. This treats the regions as ordered numbers, which is usually wrong for unordered categories:

In [ ]:
X_fit_codes = X_fit.copy()
X_val_codes = X_val.copy()
X_test_codes = X_test.copy()
for frame in (X_fit_codes, X_val_codes, X_test_codes):
    frame["region"] = frame["region"].cat.codes

codes_model = lgb.LGBMRegressor(
    n_estimators=2000, learning_rate=0.05, num_leaves=15,
    n_jobs=1, verbose=-1, random_state=5,
)
codes_model.fit(
    X_fit_codes, y_fit,
    eval_X=X_val_codes, eval_y=y_val,
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
)

print("MAE with integer codes:", round(mean_absolute_error(y_test, codes_model.predict(X_test_codes)), 4))

With only three regions, the difference is small, but with many categories the native handling matters more.

## 6. Poisson objective for claim frequency

Claim counts need the **Poisson objective**, not squared error. As in the XGBoost lesson, we model the **rate** (claims per policy-year), and pass exposure as **sample weights**. A policy in force for half a year counts half as much.

In [ ]:
rate = policies["claims"] / policies["exposure"]
exposure = policies["exposure"]

X_fit, X_val, rate_fit, rate_val, exp_fit, exp_val = train_test_split(
    train[features], rate.loc[train.index], exposure.loc[train.index],
    test_size=0.2, random_state=5,
)

freq_model = lgb.LGBMRegressor(
    objective="poisson",
    n_estimators=2000,
    learning_rate=0.05,
    num_leaves=15,
    n_jobs=1,
    verbose=-1,
    random_state=5,
)
freq_model.fit(
    X_fit, rate_fit,
    sample_weight=exp_fit,
    eval_X=X_val, eval_y=rate_val,
    eval_sample_weight=[exp_val],
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
)

print("best iteration:", freq_model.best_iteration_)

In [ ]:
pred_rate = freq_model.predict(X_test)
test_exposure = exposure.loc[test.index]

expected_claims = (pred_rate * test_exposure).sum()
actual_claims = test["claims"].sum()
print(f"expected claims: {expected_claims:,.1f}")
print(f"actual claims:   {actual_claims:,}")
print(f"A/E ratio:       {actual_claims / expected_claims:.3f}")

An A/E ratio close to 1.0 means the total number of claims is predicted correctly. This is the same check used in the GLM and XGBoost lessons, so you can compare the libraries directly.

## 7. Gamma objective for claim severity

Costs per claim are positive and right-skewed, so we use the **Gamma objective**. We build one row per claim, then simulate each cost from its own features.

In [ ]:
claims = policies.loc[policies.index.repeat(policies["claims"])].reset_index(drop=True)

claims["cost"] = rng.gamma(
    shape=2.0,
    scale=np.exp(8.0 + 0.004 * (claims["age"] - 40) + 0.3 * claims["smoker"]) / 2.0,
)

print("claim rows:", len(claims))
claims[["age", "smoker", "region", "cost"]].head()

In [ ]:
sev_train, sev_test = train_test_split(claims, test_size=0.25, random_state=5)

sev_model = lgb.LGBMRegressor(
    objective="gamma",
    n_estimators=2000,
    learning_rate=0.05,
    num_leaves=15,
    n_jobs=1,
    verbose=-1,
    random_state=5,
)
sev_model.fit(
    sev_train[features], sev_train["cost"],
    eval_X=sev_test[features], eval_y=sev_test["cost"],
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
)

print("best iteration:", sev_model.best_iteration_)
print("mean actual cost (test):   ", round(sev_test["cost"].mean(), 0))
print("mean predicted cost (test):", round(sev_model.predict(sev_test[features]).mean(), 0))

The `policies.loc[policies.index.repeat(...)]` line builds the claim table without a Python loop. `Index.repeat` repeats each policy's row by its claim count. This is the vectorized version of the `iterrows` approach in the XGBoost lesson.

## 8. Monotonic constraints

Monotonic constraints force a feature's effect to move in one direction, which is useful when an expert knows the direction. In LightGBM you give one value per feature, in the same order as the columns:

- `1` means the prediction must not decrease as the feature increases.
- `-1` means it must not increase.
- `0` means no constraint.

In [ ]:
# Constraints in feature order: age, bmi, smoker, region
constraints = [0, 1, 0, 0]   # predictions must not decrease with BMI

constrained = lgb.LGBMRegressor(
    objective="poisson",
    n_estimators=2000,
    learning_rate=0.05,
    num_leaves=15,
    monotone_constraints=constraints,
    n_jobs=1,
    verbose=-1,
    random_state=5,
)
constrained.fit(
    X_fit, rate_fit,
    sample_weight=exp_fit,
    eval_X=X_val, eval_y=rate_val,
    eval_sample_weight=[exp_val],
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
)

# Check: holding other features fixed, predictions should never fall as BMI rises
probe = X_test.iloc[[0]].copy()
bmi_grid = np.linspace(15, 50, 15)
probe_rows = pd.concat([probe] * len(bmi_grid), ignore_index=True)
probe_rows["bmi"] = bmi_grid
curve = constrained.predict(probe_rows)

print("non-decreasing in BMI:", bool(np.all(np.diff(curve) >= -1e-9)))

Like XGBoost, LightGBM uses monotonic constraints to encode domain knowledge. Use them when you know the direction, and skip them otherwise.

## 9. Feature importance

LightGBM reports importance in two ways. `"split"` counts how often a feature is used in a split, and `"gain"` measures the total improvement in the loss from those splits. Gain is usually more informative.

In [ ]:
importance = pd.DataFrame({
    "split": freq_model.booster_.feature_importance(importance_type="split"),
    "gain": freq_model.booster_.feature_importance(importance_type="gain"),
}, index=features)

importance.sort_values("gain", ascending=False).round(1)

Like the other libraries, importance tells you how much the model uses a feature, not whether the effect is causal or which direction it points.

## 10. The native API

LightGBM's native API uses a `Dataset` object and `lgb.train`. It gives finer control, and it's what most LightGBM documentation uses. Categorical columns are detected automatically from their pandas dtype:

In [ ]:
dtrain = lgb.Dataset(X_fit, label=rate_fit, weight=exp_fit)
dval = lgb.Dataset(X_val, label=rate_val, weight=exp_val, reference=dtrain)

params = {
    "objective": "poisson",
    "learning_rate": 0.05,
    "num_leaves": 15,
    "num_threads": 1,
    "verbose": -1,
    "seed": 5,
}

booster = lgb.train(
    params,
    dtrain,
    num_boost_round=2000,
    valid_sets=[dval],
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
)

print("best iteration:", booster.best_iteration)

`num_threads` is the native name for `n_jobs`. The `reference=dtrain` argument ties the validation set to the training set's category codes, so the same region maps to the same number in both.

## 11. Saving and loading

LightGBM saves models as a plain-text file, which is easy to inspect. Like XGBoost, this avoids the risks of pickling:

In [ ]:
import tempfile
from pathlib import Path

model_path = Path(tempfile.gettempdir()) / "claims_frequency_lgb.txt"
booster.save_model(model_path)

reloaded = lgb.Booster(model_file=model_path)

same = np.allclose(reloaded.predict(X_test), booster.predict(X_test))
print("predictions match:", same)

## 12. Practice exercises

Try each exercise before you look at the solution. Cells marked `# TODO` are for you to fill in.

**Exercise 1.** Change `num_leaves` in the frequency model to 4 and to 63. Which gives the lower test MAE on rates? Explain why LightGBM controls complexity with `num_leaves` rather than `max_depth`.

In [ ]:
# TODO

<details><summary>Solution</summary>

```python
from sklearn.metrics import mean_absolute_error

test_rate = rate.loc[test.index]
for leaves in [4, 15, 63]:
    m = lgb.LGBMRegressor(
        objective="poisson", n_estimators=2000, learning_rate=0.05,
        num_leaves=leaves, n_jobs=1, verbose=-1, random_state=5,
    )
    m.fit(X_fit, rate_fit, sample_weight=exp_fit,
          eval_X=X_val, eval_y=rate_val, eval_sample_weight=[exp_val],
          callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)])
    print(leaves, round(mean_absolute_error(test_rate, m.predict(X_test)), 4))
```

LightGBM grows trees leaf by leaf, choosing the best split anywhere in the tree. Depth alone doesn't limit that well, so `num_leaves` is the direct control over how many regions a tree can carve up.

</details>

**Exercise 2.** Compare the `gain` and `split` importances for `region`. Does `region` look more important by one measure than the other?

In [ ]:
# TODO

<details><summary>Solution</summary>

```python
importance.loc["region"]
```

`split` counts how often a feature appears in the trees, and `gain` measures how much it improved the fit. A feature can be used often for small gains (high split count, low gain), or used rarely for large gains. Gain is the better measure of real influence.

</details>

**Exercise 3.** Refit the severity model with the Tweedie objective (`objective="tweedie"`, `tweedie_variance_power=1.5`) instead of Gamma. Does the mean predicted cost change?

In [ ]:
# TODO

<details><summary>Solution</summary>

```python
tweedie_sev = lgb.LGBMRegressor(
    objective="tweedie",
    tweedie_variance_power=1.5,
    n_estimators=2000,
    learning_rate=0.05,
    num_leaves=15,
    n_jobs=1,
    verbose=-1,
    random_state=5,
)
tweedie_sev.fit(
    sev_train[features], sev_train["cost"],
    eval_X=sev_test[features], eval_y=sev_test["cost"],
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
)
print("mean predicted cost (tweedie):", round(tweedie_sev.predict(sev_test[features]).mean(), 0))
```

Tweedie is a family that includes Poisson and Gamma. Its variance power sets where it sits in that range (between 1 and 2 blends count-like and cost-like behavior). Compare the mean prediction to the Gamma model's value from the lesson.

</details>

## Summary

- **`LGBMRegressor`** follows the same `fit`/`predict` pattern as the other libraries.
- **`num_leaves`** is the main complexity control, since trees grow leaf-wise.
- **Early stopping** is a callback (`lgb.early_stopping`).
- **Categorical columns** work natively with the pandas `category` dtype.
- **`poisson`** with exposure as weights models claim frequency. **`gamma`** and **`tweedie`** model costs.
- **Monotonic constraints** take one value per feature, in column order.
- **Gain** importance is usually more informative than split counts.
- **Save models as text** with `save_model`, not pickle.